In [46]:
!pip install emoji pyarabic nltk

In [47]:
## Preprocessing and Cleaning:
import nltk
import re
import emoji
import pandas as pd
import pyarabic.araby as araby
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer
from nltk.tokenize import word_tokenize

## Text Representation:
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


nltk.download('punkt')        # ---------> 'punkt' is a pre-trained NLTK tokenizer model that enables correct splitting of text into words or sentences.
nltk.download('stopwords')    # ---------> 'stopwords' is an NLTK resource that provides lists of common words (like “the”, “and”) to filter out during text preprocessing.
nltk.download('punkt_tab')    # ---------> 'punkt_tab' is a pre-trained NLTK tokenizer model that allows splitting text into words or sentences for tokenization.

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [48]:
df=pd.read_csv('AAFAQ_Dataset.csv')
df.head()

,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.


### **Exploratory Data Analysis**

In [49]:
#Dataset size
df.size

15027

In [50]:
#Dataset shape
df.shape

(5009, 3)

In [51]:
#olumn names
df.columns

Index(['QuestionText', 'Category', 'Answer'], dtype='object')

In [52]:
#Data types
df.dtypes

,0
QuestionText,object
Category,object
Answer,object


In [53]:
#Labels
df['Category'].unique()

array(['التعليم', 'الاقتصاد والعمل', 'الصحة', 'التكنولوجيا',
       'البيئة والطاقة', 'الدين', 'الثقافة', 'الجغرافيا',
       'السياسة والقانون', 'التاريخ', 'العلوم', 'الترفيه', 'الرياضة',
       'السفر والسياحة', 'التطوع', 'علم الاجتماع', 'البيولوجيا'],
      dtype=object)

In [54]:
#Null values check
df.isnull().sum()

,0
QuestionText,0
Category,0
Answer,0


In [55]:
#Duplicates Check
df.duplicated().sum()

np.int64(188)

In [56]:
#Duplicated questions
df.duplicated(subset=["QuestionText"]).sum()

np.int64(364)

In [57]:
#Duplicated answers
df.duplicated(subset=["Answer"]).sum()

np.int64(415)

In [58]:
#Duplicates dropping
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [59]:
df.duplicated(subset=["QuestionText"]).sum()

np.int64(176)

In [60]:
df.duplicated(subset=["Answer"]).sum()

np.int64(227)

In [61]:
duplicates = df.groupby('QuestionText')['Answer'].nunique()

conflicting = duplicates[duplicates > 1].index

result = df[df['QuestionText'].isin(conflicting)].sort_values('QuestionText')

print(f"Number of questions with conflicting answers: {len(conflicting)}")
print()
print(result[['QuestionText', 'Answer']])

Number of questions with conflicting answers: 135

                                          QuestionText  \
1079       أعرب عن وجهة نظرك في دور المرأة في المجتمع.   
1099       أعرب عن وجهة نظرك في دور المرأة في المجتمع.   
3354  أي الصحابة قادوا الجيوش الإسلامية في حروب الردة؟   
3156  أي الصحابة قادوا الجيوش الإسلامية في حروب الردة؟   
3657                        أين تقع الأهرامات المصرية؟   
...                                                ...   
128                      هل استطاع المسلمون فتح القدس؟   
126           هل تمت توسعة المسجد النبوي في عهد عثمان؟   
124           هل تمت توسعة المسجد النبوي في عهد عثمان؟   
125          هل جمع زيد بن ثابت القرآن في عهد أبي بكر؟   
123          هل جمع زيد بن ثابت القرآن في عهد أبي بكر؟   

                                                 Answer  
1079  أرى أن دور المرأة في المجتمع حيوي وأساسي، فهي ...  
1099  أرى أن المرأة تلعب دوراً حيوياً في تنمية المجت...  
3354  أرسل الجيوش الإسلاميّة بقيادة عكرمة بن أبي جهل...  
3156  أرسل الجيوش ال

In [62]:
#keep longest answer of duplicated questions
df= df.loc[df.groupby('QuestionText')['Answer'].apply(lambda x: x.str.len().idxmax())]

In [63]:
df['QuestionText'].apply(type).value_counts()

,count
QuestionText,
<class 'str'>,4645


In [64]:
df['Answer'].apply(type).value_counts()

,count
Answer,
<class 'str'>,4645


In [65]:
Columns=['QuestionText','Answer']

In [66]:
has_english_q = df['QuestionText'].str.contains(r'[A-Za-z]', regex=True).sum()
has_english_q

np.int64(22)

In [67]:
has_english_a = df['Answer'].str.contains(r'[A-Za-z]', regex=True).sum()
has_english_a

np.int64(131)

In [68]:
has_numbers_q = df['QuestionText'].str.contains(r'\d', regex=True).sum()
has_numbers_q

np.int64(130)

In [69]:
has_numbers_a = df['Answer'].str.contains(r'\d', regex=True).sum()
has_numbers_a

np.int64(531)

# **Emoji Removal**

In [70]:
import emoji

def has_emoji(text):
    return any(char in emoji.EMOJI_DATA for char in str(text))

emoji_q = df[df['QuestionText'].apply(has_emoji)]
emoji_a = df[df['Answer'].apply(has_emoji)]

print(f"Questions with emojis: {len(emoji_q)}")
print(f"Answers with emojis: {len(emoji_a)}")

Questions with emojis: 0
Answers with emojis: 0


# **Translate English text**

In [71]:
!pip install pyarabic deep_translator

In [72]:
from deep_translator import GoogleTranslator
translator = GoogleTranslator(source='en', target='ar')

def translate_english_to_arabic(text):
    english_words = re.findall(r'[A-Za-z]+', text)
    for word in english_words:
        try:
          translated=translator.translate(word)
          text= text.replace(word,translated)
        except:
          continue
    return text

df['QuestionText'] = df['QuestionText'].apply(translate_english_to_arabic)
df['Answer'] = df['Answer'].apply(translate_english_to_arabic)
df

,QuestionText,Category,Answer
2307,أالتمرينات الرياضية اليومية تساعد في تحسين الص...,الصحة,التمرينات الرياضية اليومية تساهم بشكل كبير في ...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.
1076,أعرب عن اهتمامك بالبيئة.,الثقافة,الحفاظ على البيئة ضروري لضمان استدامة الموارد ...
1074,أعرب عن اهتمامك بالتعلم المستمر.,التعليم,التعلم المستمر يعزز من النمو الشخصي والمهني، و...
...,...,...,...
3346,‏اعط مثالين عن الأدوية القاتلة للخلايا أو المث...,الصحة,الأدوية القاتلة للخلايا مثل دواء السيكلوفوسفام...
3347,‏اعطِ مثالين عن الفقاريات الفكية البدائية.,التعليم,الفقاريات الفكية البدائية تشمل الأسماك المدرعة...
3694,‏اعطِ مثالًا عن الطفيلي الذي استخدم استراتيجية...,الصحة,الطفيلي المسبب لمرض النوم الأفريقي، تريبانوسوم...
3692,‏اعطِ مثالًا عن عقار مثبط للمناعة يمنع نشاط ال...,الصحة,مثبطات المناعة مثل دواء التاكروليموس تعمل على ...


In [73]:
remaining_q = df[df['QuestionText'].str.contains(r'[A-Za-z]', regex=True)]
remaining_a = df[df['Answer'].str.contains(r'[A-Za-z]', regex=True)]

remaining_q
remaining_a

,QuestionText,Category,Answer
2313,اشرح كيف تعمل الحاسوب.,التكنولوجيا,الحاسوب يعمل عبر مجموعة من المكونات الأساسية م...
3439,عدد بعضًا من شركات إدارة الصيدليات الكبيرة.,الصحة,بعض شركات إدارة الصيدليات الكبيرة تشمل: والجري...
3812,عدِّدْ مكونات الكمبيوتر.,التكنولوجيا,مكونات الكمبيوتر تشمل: وحدة المعالجة المركزية ...
3699,على أي مقياس سيُظهر العلماء قياسات الغطاء النب...,البيئة والطاقة,سيُظهر العلماء قياسات الغطاء النباتي على مقياس...
2929,كم عدد الشركات المدرجة في بورصة WSE في أغسطس 2...,الاقتصاد والعمل,عدد الشركات المدرجة في بورصة WSE في أغسطس 2009...
2207,كيف يتم تشخيص الفصام؟,الصحة,يتم تشخيص الفصام من خلال تقييم الأعراض النفسية...
2200,كيف يعمل محرك أقراص الحالة الصلبة؟,التكنولوجيا,يعمل محرك أقراص الحالة الصلبة (SSD) عن طريق تخ...
4141,ما الرياضة التي ينتمي إليها فريق لوس أنجلس آنج...,الرياضة,الرياضة التي ينتمي إليها فريق لوس أنجلس آنجيلز...
4127,ما هو الافتراض غير المثبت الذي يُعزى عموماً إل...,التكنولوجيا,الافتراض غير المثبت الذي يُعزى عموماً إلى قيمة...
4128,ما هو التعبير الذي يمكن استخدامه لتوضيح عدم ال...,التكنولوجيا,"التعبير الذي يمكن استخدامه هو ""ص≠Nص""، والذي يش..."


In [74]:
remaining_q


,QuestionText,Category,Answer
551,أي من الأجهزة التالية هو الأفضل للألعاب الإلكت...,التكنولوجيا,من بين الأجهزة الأفضل للألعاب الإلكترونية هو ج...
2929,كم عدد الشركات المدرجة في بورصة WSE في أغسطس 2...,الاقتصاد والعمل,عدد الشركات المدرجة في بورصة WSE في أغسطس 2009...


In [75]:
remaining_a

,QuestionText,Category,Answer
2313,اشرح كيف تعمل الحاسوب.,التكنولوجيا,الحاسوب يعمل عبر مجموعة من المكونات الأساسية م...
3439,عدد بعضًا من شركات إدارة الصيدليات الكبيرة.,الصحة,بعض شركات إدارة الصيدليات الكبيرة تشمل: والجري...
3812,عدِّدْ مكونات الكمبيوتر.,التكنولوجيا,مكونات الكمبيوتر تشمل: وحدة المعالجة المركزية ...
3699,على أي مقياس سيُظهر العلماء قياسات الغطاء النب...,البيئة والطاقة,سيُظهر العلماء قياسات الغطاء النباتي على مقياس...
2929,كم عدد الشركات المدرجة في بورصة WSE في أغسطس 2...,الاقتصاد والعمل,عدد الشركات المدرجة في بورصة WSE في أغسطس 2009...
2207,كيف يتم تشخيص الفصام؟,الصحة,يتم تشخيص الفصام من خلال تقييم الأعراض النفسية...
2200,كيف يعمل محرك أقراص الحالة الصلبة؟,التكنولوجيا,يعمل محرك أقراص الحالة الصلبة (SSD) عن طريق تخ...
4141,ما الرياضة التي ينتمي إليها فريق لوس أنجلس آنج...,الرياضة,الرياضة التي ينتمي إليها فريق لوس أنجلس آنجيلز...
4127,ما هو الافتراض غير المثبت الذي يُعزى عموماً إل...,التكنولوجيا,الافتراض غير المثبت الذي يُعزى عموماً إلى قيمة...
4128,ما هو التعبير الذي يمكن استخدامه لتوضيح عدم ال...,التكنولوجيا,"التعبير الذي يمكن استخدامه هو ""ص≠Nص""، والذي يش..."


In [76]:
df['QuestionText'][551]

'أي من الأجهزة التالية هو الأفضل للألعاب الإلكترونية: بلاي ستيشن 5، اكس بوكس مسلسل X، ألعاب الكمبيوتر الشخصي؟'

In [77]:
df['QuestionText'][2929]

'كم عدد الشركات المدرجة في بورصة WSE في أغسطس 2009؟'

In [78]:
df['Answer'][2313]

'الحاسوب يعمل عبر مجموعة من المكونات الأساسية مثل وحدة المعالجة المركزية (وحدة المعالجة المركزية)، الذاكرة (كبش)، ووحدات التخزين (الأقراص الصلبة, SSD)، وكل هذه المكونات تعمل معاً من خلال نظام التشغيل لتنفيذ الأوامر ومعالجة البيانات.'

# **Remove Tatweel**

In [79]:
#remove tatweel
for col in Columns:
  df[col]=df[col].apply(lambda x: araby.strip_tatweel(x))
df


,QuestionText,Category,Answer
2307,أالتمرينات الرياضية اليومية تساعد في تحسين الص...,الصحة,التمرينات الرياضية اليومية تساهم بشكل كبير في ...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.
1076,أعرب عن اهتمامك بالبيئة.,الثقافة,الحفاظ على البيئة ضروري لضمان استدامة الموارد ...
1074,أعرب عن اهتمامك بالتعلم المستمر.,التعليم,التعلم المستمر يعزز من النمو الشخصي والمهني، و...
...,...,...,...
3346,‏اعط مثالين عن الأدوية القاتلة للخلايا أو المث...,الصحة,الأدوية القاتلة للخلايا مثل دواء السيكلوفوسفام...
3347,‏اعطِ مثالين عن الفقاريات الفكية البدائية.,التعليم,الفقاريات الفكية البدائية تشمل الأسماك المدرعة...
3694,‏اعطِ مثالًا عن الطفيلي الذي استخدم استراتيجية...,الصحة,الطفيلي المسبب لمرض النوم الأفريقي، تريبانوسوم...
3692,‏اعطِ مثالًا عن عقار مثبط للمناعة يمنع نشاط ال...,الصحة,مثبطات المناعة مثل دواء التاكروليموس تعمل على ...


# **Normalize Hamza**

In [80]:
#normalize hamza
for col in Columns:
  df[col]=df[col].apply(lambda x:araby.normalize_hamza(x, method='tasheel'))
df

,QuestionText,Category,Answer
2307,االتمرينات الرياضية اليومية تساعد في تحسين الص...,الصحة,التمرينات الرياضية اليومية تساهم بشكل كبير في ...
2,اتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,اتعرف البكتيريا بانها كاينات حية دقيقة؟,التعليم,البكتيريا تُعرف بانها كاينات حية دقيقة.
1076,اعرب عن اهتمامك بالبيية.,الثقافة,الحفاظ على البيية ضروري لضمان استدامة الموارد ...
1074,اعرب عن اهتمامك بالتعلم المستمر.,التعليم,التعلم المستمر يعزز من النمو الشخصي والمهني، و...
...,...,...,...
3346,‏اعط مثالين عن الادوية القاتلة للخلايا او المث...,الصحة,الادوية القاتلة للخلايا مثل دواء السيكلوفوسفام...
3347,‏اعطِ مثالين عن الفقاريات الفكية البدايية.,التعليم,الفقاريات الفكية البدايية تشمل الاسماك المدرعة...
3694,‏اعطِ مثالًا عن الطفيلي الذي استخدم استراتيجية...,الصحة,الطفيلي المسبب لمرض النوم الافريقي، تريبانوسوم...
3692,‏اعطِ مثالًا عن عقار مثبط للمناعة يمنع نشاط ال...,الصحة,مثبطات المناعة مثل دواء التاكروليموس تعمل على ...


In [81]:
for col in Columns:
  df[col] = df[col].str.replace('ة', 'ه', regex=False)
df

,QuestionText,Category,Answer
2307,االتمرينات الرياضيه اليوميه تساعد في تحسين الص...,الصحة,التمرينات الرياضيه اليوميه تساهم بشكل كبير في ...
2,اتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,اتعرف البكتيريا بانها كاينات حيه دقيقه؟,التعليم,البكتيريا تُعرف بانها كاينات حيه دقيقه.
1076,اعرب عن اهتمامك بالبييه.,الثقافة,الحفاظ على البييه ضروري لضمان استدامه الموارد ...
1074,اعرب عن اهتمامك بالتعلم المستمر.,التعليم,التعلم المستمر يعزز من النمو الشخصي والمهني، و...
...,...,...,...
3346,‏اعط مثالين عن الادويه القاتله للخلايا او المث...,الصحة,الادويه القاتله للخلايا مثل دواء السيكلوفوسفام...
3347,‏اعطِ مثالين عن الفقاريات الفكيه البداييه.,التعليم,الفقاريات الفكيه البداييه تشمل الاسماك المدرعه...
3694,‏اعطِ مثالًا عن الطفيلي الذي استخدم استراتيجيه...,الصحة,الطفيلي المسبب لمرض النوم الافريقي، تريبانوسوم...
3692,‏اعطِ مثالًا عن عقار مثبط للمناعه يمنع نشاط ال...,الصحة,مثبطات المناعه مثل دواء التاكروليموس تعمل على ...


In [82]:
for col in Columns:
  df[col] = df[col].str.replace('ى', 'ي', regex=False)
df

,QuestionText,Category,Answer
2307,االتمرينات الرياضيه اليوميه تساعد في تحسين الص...,الصحة,التمرينات الرياضيه اليوميه تساهم بشكل كبير في ...
2,اتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,اتعرف البكتيريا بانها كاينات حيه دقيقه؟,التعليم,البكتيريا تُعرف بانها كاينات حيه دقيقه.
1076,اعرب عن اهتمامك بالبييه.,الثقافة,الحفاظ علي البييه ضروري لضمان استدامه الموارد ...
1074,اعرب عن اهتمامك بالتعلم المستمر.,التعليم,التعلم المستمر يعزز من النمو الشخصي والمهني، و...
...,...,...,...
3346,‏اعط مثالين عن الادويه القاتله للخلايا او المث...,الصحة,الادويه القاتله للخلايا مثل دواء السيكلوفوسفام...
3347,‏اعطِ مثالين عن الفقاريات الفكيه البداييه.,التعليم,الفقاريات الفكيه البداييه تشمل الاسماك المدرعه...
3694,‏اعطِ مثالًا عن الطفيلي الذي استخدم استراتيجيه...,الصحة,الطفيلي المسبب لمرض النوم الافريقي، تريبانوسوم...
3692,‏اعطِ مثالًا عن عقار مثبط للمناعه يمنع نشاط ال...,الصحة,مثبطات المناعه مثل دواء التاكروليموس تعمل علي ...


In [83]:
has_numbers_q = df['QuestionText'].str.contains(r'\d', regex=True).sum()
has_numbers_q

np.int64(130)

In [84]:
has_english_a = df['Answer'].str.contains(r'[A-Za-z]', regex=True).sum()
has_english_a

np.int64(14)

# **Remove punctuation and english letters left**

In [85]:
#remove punctuation and english letters left
import re
def remove_punct(text):
    cleaned = re.sub(r'[^\u0621-\u064A0-9\s]', '', text)
    return cleaned.strip()

for col in Columns:
  df[col]=df[col].apply(remove_punct)
df

,QuestionText,Category,Answer
2307,االتمرينات الرياضيه اليوميه تساعد في تحسين الص...,الصحة,التمرينات الرياضيه اليوميه تساهم بشكل كبير في ...
2,اتصعد الشمس من الشرق,التعليم,الشمس تصعد من الشرق
3,اتعرف البكتيريا بانها كاينات حيه دقيقه,التعليم,البكتيريا تعرف بانها كاينات حيه دقيقه
1076,اعرب عن اهتمامك بالبييه,الثقافة,الحفاظ علي البييه ضروري لضمان استدامه الموارد ...
1074,اعرب عن اهتمامك بالتعلم المستمر,التعليم,التعلم المستمر يعزز من النمو الشخصي والمهني وي...
...,...,...,...
3346,اعط مثالين عن الادويه القاتله للخلايا او المثب...,الصحة,الادويه القاتله للخلايا مثل دواء السيكلوفوسفام...
3347,اعط مثالين عن الفقاريات الفكيه البداييه,التعليم,الفقاريات الفكيه البداييه تشمل الاسماك المدرعه...
3694,اعط مثالا عن الطفيلي الذي استخدم استراتيجيه تب...,الصحة,الطفيلي المسبب لمرض النوم الافريقي تريبانوسوما...
3692,اعط مثالا عن عقار مثبط للمناعه يمنع نشاط الخلا...,الصحة,مثبطات المناعه مثل دواء التاكروليموس تعمل علي ...


In [86]:
has_numbers_q = df['QuestionText'].str.contains(r'\d', regex=True).sum()
has_numbers_q

np.int64(130)

In [87]:
has_english_q = df['QuestionText'].str.contains(r'[A-Za-z]', regex=True).sum()
has_english_q

np.int64(0)

In [88]:
has_english_a = df['Answer'].str.contains(r'[A-Za-z]', regex=True).sum()
has_english_a

np.int64(0)

In [89]:
remaining_q = df[df['QuestionText'].str.contains(r'[A-Za-z]', regex=True)]
remaining_a = df[df['Answer'].str.contains(r'[A-Za-z]', regex=True)]

remaining_q
remaining_a

,QuestionText,Category,Answer


In [90]:
remaining_q

,QuestionText,Category,Answer


# **Tashkeel**

In [91]:
df['QuestionText_MT'] = df['QuestionText'].copy() #we dont want to remove tashkeel for MT
df['Answer_MT']       = df['Answer'].copy()

In [92]:
def remove_tashkeel(text):
    if pd.isna(text):
        return text
    return re.sub(r'[\u064B-\u065F]', '', text)

In [93]:
df['QuestionText_QA'] = df['QuestionText'].apply(remove_tashkeel)
df['Answer_QA']       = df['Answer'].apply(remove_tashkeel)

# **Tokenization**

In [94]:
#tokenization for QA
df['QuestionText_QA'] = df['QuestionText_QA'].apply(word_tokenize)
df['Answer_QA']       = df['Answer_QA'].apply(word_tokenize)
df

,QuestionText,Category,Answer,QuestionText_MT,Answer_MT,QuestionText_QA,Answer_QA
2307,االتمرينات الرياضيه اليوميه تساعد في تحسين الص...,الصحة,التمرينات الرياضيه اليوميه تساهم بشكل كبير في ...,االتمرينات الرياضيه اليوميه تساعد في تحسين الص...,التمرينات الرياضيه اليوميه تساهم بشكل كبير في ...,"[االتمرينات, الرياضيه, اليوميه, تساعد, في, تحس...","[التمرينات, الرياضيه, اليوميه, تساهم, بشكل, كب..."
2,اتصعد الشمس من الشرق,التعليم,الشمس تصعد من الشرق,اتصعد الشمس من الشرق,الشمس تصعد من الشرق,"[اتصعد, الشمس, من, الشرق]","[الشمس, تصعد, من, الشرق]"
3,اتعرف البكتيريا بانها كاينات حيه دقيقه,التعليم,البكتيريا تعرف بانها كاينات حيه دقيقه,اتعرف البكتيريا بانها كاينات حيه دقيقه,البكتيريا تعرف بانها كاينات حيه دقيقه,"[اتعرف, البكتيريا, بانها, كاينات, حيه, دقيقه]","[البكتيريا, تعرف, بانها, كاينات, حيه, دقيقه]"
1076,اعرب عن اهتمامك بالبييه,الثقافة,الحفاظ علي البييه ضروري لضمان استدامه الموارد ...,اعرب عن اهتمامك بالبييه,الحفاظ علي البييه ضروري لضمان استدامه الموارد ...,"[اعرب, عن, اهتمامك, بالبييه]","[الحفاظ, علي, البييه, ضروري, لضمان, استدامه, ا..."
1074,اعرب عن اهتمامك بالتعلم المستمر,التعليم,التعلم المستمر يعزز من النمو الشخصي والمهني وي...,اعرب عن اهتمامك بالتعلم المستمر,التعلم المستمر يعزز من النمو الشخصي والمهني وي...,"[اعرب, عن, اهتمامك, بالتعلم, المستمر]","[التعلم, المستمر, يعزز, من, النمو, الشخصي, وال..."
...,...,...,...,...,...,...,...
3346,اعط مثالين عن الادويه القاتله للخلايا او المثب...,الصحة,الادويه القاتله للخلايا مثل دواء السيكلوفوسفام...,اعط مثالين عن الادويه القاتله للخلايا او المثب...,الادويه القاتله للخلايا مثل دواء السيكلوفوسفام...,"[اعط, مثالين, عن, الادويه, القاتله, للخلايا, ا...","[الادويه, القاتله, للخلايا, مثل, دواء, السيكلو..."
3347,اعط مثالين عن الفقاريات الفكيه البداييه,التعليم,الفقاريات الفكيه البداييه تشمل الاسماك المدرعه...,اعط مثالين عن الفقاريات الفكيه البداييه,الفقاريات الفكيه البداييه تشمل الاسماك المدرعه...,"[اعط, مثالين, عن, الفقاريات, الفكيه, البداييه]","[الفقاريات, الفكيه, البداييه, تشمل, الاسماك, ا..."
3694,اعط مثالا عن الطفيلي الذي استخدم استراتيجيه تب...,الصحة,الطفيلي المسبب لمرض النوم الافريقي تريبانوسوما...,اعط مثالا عن الطفيلي الذي استخدم استراتيجيه تب...,الطفيلي المسبب لمرض النوم الافريقي تريبانوسوما...,"[اعط, مثالا, عن, الطفيلي, الذي, استخدم, استرات...","[الطفيلي, المسبب, لمرض, النوم, الافريقي, تريبا..."
3692,اعط مثالا عن عقار مثبط للمناعه يمنع نشاط الخلا...,الصحة,مثبطات المناعه مثل دواء التاكروليموس تعمل علي ...,اعط مثالا عن عقار مثبط للمناعه يمنع نشاط الخلا...,مثبطات المناعه مثل دواء التاكروليموس تعمل علي ...,"[اعط, مثالا, عن, عقار, مثبط, للمناعه, يمنع, نش...","[مثبطات, المناعه, مثل, دواء, التاكروليموس, تعم..."


# **Lemmatization**

In [95]:
from nltk.stem.isri import ISRIStemmer

stemmer = ISRIStemmer()

df['QuestionText_lemmatized'] = df['QuestionText_QA'].apply(lambda tokens: [stemmer.stem(token) for token in tokens])
df['Answer_lemmatized']       = df['Answer_QA'].apply(lambda tokens: [stemmer.stem(token) for token in tokens])
df

,QuestionText,Category,Answer,QuestionText_MT,Answer_MT,QuestionText_QA,Answer_QA,QuestionText_lemmatized,Answer_lemmatized
2307,االتمرينات الرياضيه اليوميه تساعد في تحسين الص...,الصحة,التمرينات الرياضيه اليوميه تساهم بشكل كبير في ...,االتمرينات الرياضيه اليوميه تساعد في تحسين الص...,التمرينات الرياضيه اليوميه تساهم بشكل كبير في ...,"[االتمرينات, الرياضيه, اليوميه, تساعد, في, تحس...","[التمرينات, الرياضيه, اليوميه, تساهم, بشكل, كب...","[االتمرين, ريض, يوم, سعد, في, تحس, صحه, عمه]","[مرن, ريض, يوم, تسا, شكل, كبر, في, تحس, صحه, ع..."
2,اتصعد الشمس من الشرق,التعليم,الشمس تصعد من الشرق,اتصعد الشمس من الشرق,الشمس تصعد من الشرق,"[اتصعد, الشمس, من, الشرق]","[الشمس, تصعد, من, الشرق]","[صعد, شمس, من, شرق]","[شمس, صعد, من, شرق]"
3,اتعرف البكتيريا بانها كاينات حيه دقيقه,التعليم,البكتيريا تعرف بانها كاينات حيه دقيقه,اتعرف البكتيريا بانها كاينات حيه دقيقه,البكتيريا تعرف بانها كاينات حيه دقيقه,"[اتعرف, البكتيريا, بانها, كاينات, حيه, دقيقه]","[البكتيريا, تعرف, بانها, كاينات, حيه, دقيقه]","[عرف, كتر, بان, كين, حيه, دقق]","[كتر, عرف, بان, كين, حيه, دقق]"
1076,اعرب عن اهتمامك بالبييه,الثقافة,الحفاظ علي البييه ضروري لضمان استدامه الموارد ...,اعرب عن اهتمامك بالبييه,الحفاظ علي البييه ضروري لضمان استدامه الموارد ...,"[اعرب, عن, اهتمامك, بالبييه]","[الحفاظ, علي, البييه, ضروري, لضمان, استدامه, ا...","[عرب, عن, همم, بيه]","[حفظ, علي, بيه, ضرر, لضم, دام, ورد, طبع, جيل, ..."
1074,اعرب عن اهتمامك بالتعلم المستمر,التعليم,التعلم المستمر يعزز من النمو الشخصي والمهني وي...,اعرب عن اهتمامك بالتعلم المستمر,التعلم المستمر يعزز من النمو الشخصي والمهني وي...,"[اعرب, عن, اهتمامك, بالتعلم, المستمر]","[التعلم, المستمر, يعزز, من, النمو, الشخصي, وال...","[عرب, عن, همم, علم, سمر]","[علم, سمر, عزز, من, نمو, شخص, هني, سعد, في, تك..."
...,...,...,...,...,...,...,...,...,...
3346,اعط مثالين عن الادويه القاتله للخلايا او المثب...,الصحة,الادويه القاتله للخلايا مثل دواء السيكلوفوسفام...,اعط مثالين عن الادويه القاتله للخلايا او المثب...,الادويه القاتله للخلايا مثل دواء السيكلوفوسفام...,"[اعط, مثالين, عن, الادويه, القاتله, للخلايا, ا...","[الادويه, القاتله, للخلايا, مثل, دواء, السيكلو...","[اعط, ثال, عن, ادي, قتل, خلا, او, ثبط, نعه]","[ادي, قتل, خلا, مثل, دوء, سيكلوفوسفاميد, ادي, ..."
3347,اعط مثالين عن الفقاريات الفكيه البداييه,التعليم,الفقاريات الفكيه البداييه تشمل الاسماك المدرعه...,اعط مثالين عن الفقاريات الفكيه البداييه,الفقاريات الفكيه البداييه تشمل الاسماك المدرعه...,"[اعط, مثالين, عن, الفقاريات, الفكيه, البداييه]","[الفقاريات, الفكيه, البداييه, تشمل, الاسماك, ا...","[اعط, ثال, عن, فقر, فكه, بدي]","[فقر, فكه, بدي, شمل, سمك, درع, سمك, عظم, بدي, ..."
3694,اعط مثالا عن الطفيلي الذي استخدم استراتيجيه تب...,الصحة,الطفيلي المسبب لمرض النوم الافريقي تريبانوسوما...,اعط مثالا عن الطفيلي الذي استخدم استراتيجيه تب...,الطفيلي المسبب لمرض النوم الافريقي تريبانوسوما...,"[اعط, مثالا, عن, الطفيلي, الذي, استخدم, استرات...","[الطفيلي, المسبب, لمرض, النوم, الافريقي, تريبا...","[اعط, ثلا, عن, طفل, الذي, خدم, استراتيجيه, تبا...","[طفل, سبب, مرض, نوم, فرق, تريبانوسو, برس, خدم,..."
3692,اعط مثالا عن عقار مثبط للمناعه يمنع نشاط الخلا...,الصحة,مثبطات المناعه مثل دواء التاكروليموس تعمل علي ...,اعط مثالا عن عقار مثبط للمناعه يمنع نشاط الخلا...,مثبطات المناعه مثل دواء التاكروليموس تعمل علي ...,"[اعط, مثالا, عن, عقار, مثبط, للمناعه, يمنع, نش...","[مثبطات, المناعه, مثل, دواء, التاكروليموس, تعم...","[اعط, ثلا, عن, عقر, ثبط, نعه, منع, نشط, خلا, ا...","[ثبط, نعه, مثل, دوء, تاكروليموس, عمل, علي, منع..."


In [96]:
df.to_csv("NLP_dataset.csv", index=False, encoding="utf-8-sig")

In [97]:
from google.colab import files
files.download("NLP_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>